# DevFlow - Lab: Busca no Azure AI Search

**Curso:** Agentic Engineering - SkillGo
**Prof.:** Ives Santos

---

Consulta o indice `devflow-kb` e mostra os trechos da base de conhecimento mais parecidos com uma pergunta.

## 1. Ambiente

Crie nos **Secrets** do Colab (icone de chave na barra lateral) e ative o acesso do notebook:

| Segredo | Valor |
|---|---|
| `AZURE_SEARCH_ENDPOINT` | `https://<seu-servico>.search.windows.net` |
| `AZURE_SEARCH_INDEX` | `devflow-kb` |
| `AZURE_SEARCH_KEY` | uma chave de **consulta** (Settings > Keys > Query keys) |

In [ ]:
!pip install -q "azure-search-documents>=11.6"

In [ ]:
from google.colab import userdata

ENDPOINT = userdata.get("AZURE_SEARCH_ENDPOINT").strip()
INDICE = userdata.get("AZURE_SEARCH_INDEX").strip()
CHAVE = userdata.get("AZURE_SEARCH_KEY").strip()

print("servico:", ENDPOINT)
print("indice :", INDICE)

## 2. Conferir o indice e os campos

Busca **um** documento qualquer (`"*"`) sem escolher campos: o Azure devolve todos os campos
recuperaveis. Se algum campo esperado faltar, o notebook esta apontando para o indice ou servico errado.

In [ ]:
from azure.core.credentials import AzureKeyCredential
from azure.search.documents import SearchClient

cliente = SearchClient(ENDPOINT, INDICE, AzureKeyCredential(CHAVE))

amostra = next(iter(cliente.search(search_text="*", top=1)), None)
assert amostra is not None, "indice vazio: confira se o indexador rodou"

ESPERADOS = ["content", "content_vector", "titulo", "titulo_documento",
             "ordinal_position", "metadata_storage_name", "categoria", "dominio"]

for campo in ESPERADOS:
    print(f"  {'ok' if campo in amostra else 'FALTA':<6} {campo}")

print("\ndocumentos no indice:", cliente.get_document_count())

## 3. Classe que consulta o Azure AI Search

`buscar` faz uma busca **hibrida**:

- `search_text` - busca por palavras (BM25);
- `vector_queries` - busca por significado. A pergunta vai como **texto** e o vetorizador do indice
  a transforma em vetor.

O Azure combina as duas listas e devolve os `k` melhores. O filtro `titulo ne ''` descarta os documentos
vazios gerados pelo `#` do topo de cada arquivo.

In [ ]:
from azure.search.documents.models import VectorizableTextQuery


class BuscaAzure:
    def __init__(self, endpoint: str, indice: str, chave: str):
        self.cliente = SearchClient(endpoint, indice, AzureKeyCredential(chave))

    def buscar(self, pergunta: str, k: int = 3, filtro: str | None = None) -> list[dict]:
        filtro_final = "titulo ne ''" + (f" and {filtro}" if filtro else "")

        resultados = self.cliente.search(
            search_text=pergunta,
            vector_queries=[VectorizableTextQuery(text=pergunta, k_nearest_neighbors=10, fields="content_vector")],
            select=["metadata_storage_name", "ordinal_position", "titulo", "content"],
            filter=filtro_final,
            top=k,
        )

        return [
            {
                "score": round(doc["@search.score"], 4),
                "arquivo": doc["metadata_storage_name"],
                "secao": doc["ordinal_position"],
                "titulo": doc["titulo"],
                "texto": doc["content"].strip(),
            }
            for doc in resultados
        ]


def mostrar(trechos: list[dict]) -> None:
    for t in trechos:
        print(f"[{t['score']}] {t['arquivo']} #{t['secao']} - {t['titulo']}")
        print(f"   {t['texto'][:120]}")
    if not trechos:
        print("nenhum resultado")


busca = BuscaAzure(ENDPOINT, INDICE, CHAVE)

## 4. Teste semantico

As perguntas **nao usam as palavras** dos documentos. Mesmo assim, a parte vetorial encontra os trechos
com o mesmo significado:

| Pergunta | Trecho esperado |
|---|---|
| desconto aplicado em dobro | Aplicacao de cupons / incidente de desconto cumulativo |
| como liberar a correcao aos poucos | Feature flags de preco |
| o que falta para considerar a tarefa terminada | Definicao de pronto |

In [ ]:
perguntas = [
    "desconto aplicado em dobro",
    "como liberar a correcao aos poucos",
    "o que falta para considerar a tarefa terminada",
]

for pergunta in perguntas:
    print(f">>> {pergunta}")
    mostrar(busca.buscar(pergunta))
    print()

## 5. Busca com filtro

O `filtro` e uma expressao OData sobre campos `filterable`. Os valores de `categoria` e `dominio` vem dos
metadados dos blobs.

| Categoria | Arquivo |
|---|---|
| `politica` | politica-de-engenharia.md |
| `arquitetura` | arquitetura-checkout.md |
| `testes` | padroes-de-testes.md |
| `runbook` | runbook-pagamentos.md |
| `incidentes` | historico-incidentes.md |

A mesma pergunta, sem e com filtro:

In [ ]:
pergunta = "erro de calculo no valor cobrado"

print(">>> sem filtro")
mostrar(busca.buscar(pergunta))

print("\n>>> so a politica de engenharia")
mostrar(busca.buscar(pergunta, filtro="categoria eq 'politica'"))

print("\n>>> so o dominio checkout")
mostrar(busca.buscar(pergunta, filtro="dominio eq 'checkout'"))